# Week 2 — Data Pipeline + Identity Gate + IF Baseline

**Inputs:** `data/raw/lemd_jan2024.parquet` (from Week 1)

**Outputs:**
- `data/processed/trajectories.parquet` — segmented, interpolated, normalized
- `data/processed/train.parquet`, `val.parquet`, `test.parquet`
- `data/processed/aircraft_db.parquet` — ICAO24 lookup table
- `models/isolation_forest.pkl` — IF baseline

**Deliverable:** IF AUROC reported at end of this notebook.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/drone-ai-saturdays'
    !pip install -q scikit-learn pandas pyarrow
else:
    BASE = '..'

import os
DATA_RAW = f'{BASE}/data/raw'
DATA_PROC = f'{BASE}/data/processed'
MODELS = f'{BASE}/models'
os.makedirs(DATA_PROC, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2
import pickle
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

## 1. Trajectory Segmentation

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = radians(lat1), radians(lat2)
    dp, dl = radians(lat2-lat1), radians(lon2-lon1)
    a = sin(dp/2)**2 + cos(p1)*cos(p2)*sin(dl/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1-a))

def segment_trajectories(df, gap_time=60, gap_dist=5000, min_steps=10):
    """Split ICAO24 tracks into segments. Returns list of DataFrames."""
    segments = []
    for icao24, grp in df.sort_values('time').groupby('icao24'):
        grp = grp.reset_index(drop=True)
        seg_id = 0
        seg_start = 0
        for i in range(1, len(grp)):
            dt = grp.loc[i, 'time'] - grp.loc[i-1, 'time']
            try:
                dist = haversine(grp.loc[i-1,'lat'], grp.loc[i-1,'lon'],
                                 grp.loc[i,'lat'], grp.loc[i,'lon'])
            except:
                dist = 0
            if dt > gap_time or dist > gap_dist:
                seg = grp.iloc[seg_start:i].copy()
                seg['seg_id'] = f'{icao24}_{seg_id}'
                if len(seg) >= min_steps:
                    segments.append(seg)
                seg_id += 1
                seg_start = i
        # last segment
        seg = grp.iloc[seg_start:].copy()
        seg['seg_id'] = f'{icao24}_{seg_id}'
        if len(seg) >= min_steps:
            segments.append(seg)
    return segments

df_raw = pd.read_parquet(f'{DATA_RAW}/lemd_jan2024.parquet')
segments = segment_trajectories(df_raw)
print(f'Segments: {len(segments):,} (from {df_raw["icao24"].nunique():,} tracks)')

## 2. Feature Engineering

In [ ]:
LEMD_LAT, LEMD_LON = 40.4719, -3.5626  # LEMD ARP

def extract_features(seg):
    seg = seg.copy()
    seg['dist_lemd'] = seg.apply(
        lambda r: haversine(r['lat'], r['lon'], LEMD_LAT, LEMD_LON), axis=1)
    hour = pd.to_datetime(seg['time'], unit='s').dt.hour
    seg['tod_sin'] = np.sin(2 * np.pi * hour / 24)
    seg['tod_cos'] = np.cos(2 * np.pi * hour / 24)
    alt = seg['baroaltitude'].fillna(seg.get('geoaltitude', np.nan))
    seg['alt'] = alt.interpolate()
    seg['speed'] = seg['velocity'].interpolate()
    seg['heading'] = seg['heading'].interpolate()
    return seg[['lat','lon','alt','speed','heading','dist_lemd','tod_sin','tod_cos','seg_id']]

all_segs = pd.concat([extract_features(s) for s in segments], ignore_index=True)
print(f'Feature matrix: {all_segs.shape}')
all_segs.head()

In [ ]:
FEATURE_COLS = ['lat','lon','alt','speed','heading','dist_lemd','tod_sin','tod_cos']

# Compute normalization stats on training set only (split first)
seg_ids = all_segs['seg_id'].unique()
np.random.seed(42)
np.random.shuffle(seg_ids)
n = len(seg_ids)
train_ids = set(seg_ids[:int(0.8*n)])
val_ids   = set(seg_ids[int(0.8*n):int(0.9*n)])
test_ids  = set(seg_ids[int(0.9*n):])

train_mask = all_segs['seg_id'].isin(train_ids)
mu = all_segs.loc[train_mask, FEATURE_COLS].mean()
sigma = all_segs.loc[train_mask, FEATURE_COLS].std().replace(0, 1)

all_segs[FEATURE_COLS] = (all_segs[FEATURE_COLS] - mu) / sigma

# Save normalization stats
pd.DataFrame({'mean': mu, 'std': sigma}).to_parquet(f'{DATA_PROC}/norm_stats.parquet')

# Save splits
for split, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    all_segs[all_segs['seg_id'].isin(ids)].to_parquet(f'{DATA_PROC}/{split}.parquet', index=False)
    print(f'{split}: {len(ids):,} segments')

## 3. Identity Gate — ICAO24 Lookup

In [ ]:
# Download OpenSky aircraft DB (do once)
aircraft_db_path = f'{DATA_PROC}/aircraft_db.parquet'
if not os.path.exists(aircraft_db_path):
    import urllib.request
    print('Downloading OpenSky aircraft DB (~30MB)...')
    urllib.request.urlretrieve(
        'https://opensky-network.org/datasets/metadata/aircraftDatabase.csv',
        '/tmp/aircraftDatabase.csv')
    db = pd.read_csv('/tmp/aircraftDatabase.csv', usecols=['icao24','registration','manufacturername','model','operatorcallsign'])
    db['icao24'] = db['icao24'].str.strip().str.lower()
    db.to_parquet(aircraft_db_path, index=False)
    print(f'Saved {len(db):,} entries')

aircraft_db = pd.read_parquet(aircraft_db_path).set_index('icao24')

def identity_gate(icao24: str) -> str:
    """Returns 'CLEARED' or 'UNIDENTIFIED'."""
    if icao24.lower() in aircraft_db.index:
        return 'CLEARED'
    return 'UNIDENTIFIED'

# Spot-check on a sample of ICAO24s from our dataset
sample_ids = df_raw['icao24'].unique()[:20]
results = [(i, identity_gate(i)) for i in sample_ids]
cleared = sum(1 for _, r in results if r == 'CLEARED')
print(f'Sample check: {cleared}/{len(results)} cleared ({100*cleared/len(results):.0f}%)')
for icao24, status in results[:10]:
    print(f'  {icao24}: {status}')

## 4. Isolation Forest Baseline

In [ ]:
train_df = pd.read_parquet(f'{DATA_PROC}/train.parquet')
test_df  = pd.read_parquet(f'{DATA_PROC}/test.parquet')

# Per-trajectory summary statistics as features
def traj_stats(df):
    return df.groupby('seg_id')[FEATURE_COLS].agg(['mean','std','min','max']).fillna(0)

X_train = traj_stats(train_df)
X_test  = traj_stats(test_df)

# Inject synthetic anomalies into test set for evaluation
def inject_anomalies(df, n_anomalies=200):
    normal_segs = df['seg_id'].unique()
    chosen = np.random.choice(normal_segs, size=min(n_anomalies, len(normal_segs)), replace=False)
    anomalies = []
    for seg_id in chosen:
        seg = df[df['seg_id'] == seg_id].copy()
        atype = np.random.choice(['altitude', 'speed', 'hovering'])
        if atype == 'altitude':
            seg['alt'] = seg['alt'] + 3.0  # shift altitude up 3 std devs
        elif atype == 'speed':
            mid = len(seg) // 2
            seg.iloc[mid:mid+5, seg.columns.get_loc('speed')] *= 3
        else:  # hovering
            mid = len(seg) // 2
            seg.iloc[mid:mid+5, seg.columns.get_loc('speed')] = 0
        seg['seg_id'] = seg_id + '_anomaly'
        anomalies.append(seg)
    return pd.concat(anomalies)

anomaly_df = inject_anomalies(test_df)
X_anomaly = traj_stats(anomaly_df)

X_eval = pd.concat([X_test, X_anomaly])
y_eval = np.array([0]*len(X_test) + [1]*len(X_anomaly))
print(f'Eval set: {len(X_test)} normal + {len(X_anomaly)} anomalies')

In [ ]:
clf = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
clf.fit(X_train)

# decision_function: lower = more anomalous; negate for AUROC (higher = more anomalous)
scores = -clf.decision_function(X_eval)
auroc = roc_auc_score(y_eval, scores)
print(f'Isolation Forest AUROC: {auroc:.3f}')

if auroc > 0.85:
    print('PASS — AUROC > 0.85')
elif auroc > 0.70:
    print('MARGINAL — acceptable baseline, LSTM should improve')
else:
    print('FAIL — revisit feature engineering before Week 3')

# Save model
with open(f'{MODELS}/isolation_forest.pkl', 'wb') as f:
    pickle.dump(clf, f)
print(f'Model saved to {MODELS}/isolation_forest.pkl')